# Music Popularity Predictor

I wanted to see how well a song's audio characteristics (energy, danceability,
loudness, and so on) can predict how popular it becomes on Spotify, and how
much of that popularity is really just "who's the artist" rather than
anything about the track itself.

This notebook covers the full pipeline: cleaning the data, engineering a few
features I thought might matter, comparing a handful of models, tuning the
best one, and then explaining its predictions with SHAP so the results are
more than a black box. The final model and preprocessing objects are saved
at the end for the companion Streamlit app (`app.py`).

**Dataset:** Spotify track metadata with audio features (`Spotify_data.csv`)
**Target:** `Popularity` (0-100)
**Final model:** XGBoost, tuned with Optuna

A note on a bug I caught while building this: my first pass computed each
artist's average popularity using the *entire* dataset, before splitting
into train/test. That leaks test-set information into a training feature -
the model would essentially be told part of the answer in advance, which
inflates validation scores in a way that wouldn't hold up on genuinely new
data. The version below computes artist stats from the training set only.

## Setup

Standard imports, plus the extra libraries this notebook uses beyond core
`scikit-learn`: `xgboost` and `lightgbm` for the models being compared,
`optuna` for hyperparameter search, `shap` for explainability, and `mlflow`
to keep a record of each run's parameters and metrics.

If any of these aren't installed yet, uncomment and run the `pip install`
line in the cell below first.

In [ ]:
# Uncomment and run once if these aren't installed yet:
# !pip install xgboost lightgbm optuna shap mlflow -q

import warnings
warnings.filterwarnings('ignore')

import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import shap
import optuna
import mlflow
import mlflow.sklearn
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

os.makedirs('models', exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
sns.set_theme(style='darkgrid', palette='muted')
print('Imports OK')

## 1. Loading the data

I'm lowercasing the column names right away. The raw CSV uses titlecase
column headers (`Energy`, `Danceability`, ...), but keeping everything
lowercase from this point on makes the feature names consistent with what
the Streamlit app expects, and it's just easier to type.

In [ ]:
df = pd.read_csv('Spotify_data.csv')

# Drop any stray index column pandas sometimes leaves in exported CSVs
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

# Lowercase all column names for consistency across the notebook and app
df.columns = df.columns.str.strip().str.lower()

print(f'Shape: {df.shape}')
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print(f'\nMissing values:\n{missing}')
else:
    print('\nNo missing values')

df.head()

## 2. Train/test split - done early, on purpose

I'm splitting the data *before* engineering any feature that depends on the
target (`popularity`). The audio-feature engineering below (energy x valence,
etc.) is safe to do on the full dataset since it only uses each row's own
audio features. But the artist-level features - "how popular is this artist
on average?" - use the target column, so if I compute those before
splitting, the training set ends up containing information from the test
set. Splitting first and deriving artist stats from the training portion
only keeps the evaluation honest.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')

## 3. Feature engineering

**Derived audio features** - a few combinations of the raw audio features
that seemed like they might carry more signal than any single feature alone:
- `energy_valence`: high energy *and* high valence together roughly captures
  "upbeat" in a way neither feature does on its own
- `dance_energy`: a rough "club factor"
- `acoustic_energy_gap`: how far a track leans electric vs. organic
- `loudness_norm`: rescales loudness (normally in dB, roughly -60 to 0) to
  a friendlier 0-1 range
- `speech_ratio`: speechiness relative to energy - spoken-word or rap-heavy
  tracks tend to stand out here

**Artist features** - computed from the *training set only* (see note
above), then mapped onto both train and test. For artists in the test set
who never appear in training, I fall back to the training set's global
mean popularity - a reasonable prior for "unknown artist".

In [ ]:
def add_audio_features(frame):
    frame = frame.copy()
    frame['energy_valence']      = frame['energy'] * frame['valence']
    frame['dance_energy']        = frame['danceability'] * frame['energy']
    frame['acoustic_energy_gap'] = frame['energy'] - frame['acousticness']
    frame['loudness_norm']       = (frame['loudness'] + 60) / 60
    frame['speech_ratio']        = frame['speechiness'] / (frame['energy'] + 1e-6)
    return frame

train_df = add_audio_features(train_df)
test_df  = add_audio_features(test_df)

# Artist-level stats, learned from training data only
if 'artist' in train_df.columns:
    artist_stats = (
        train_df.groupby('artist')['popularity']
        .agg(artist_mean_pop='mean', artist_song_count='count')
    )
    global_mean_pop = train_df['popularity'].mean()

    train_df = train_df.merge(artist_stats, on='artist', how='left')
    test_df  = test_df.merge(artist_stats, on='artist', how='left')

    # Artists unseen during training fall back to the training-set average
    test_df['artist_mean_pop']   = test_df['artist_mean_pop'].fillna(global_mean_pop)
    test_df['artist_song_count'] = test_df['artist_song_count'].fillna(0)

    unseen = test_df['artist_song_count'].eq(0).sum()
    print(f'Artist features added. {unseen} test-set rows are unseen artists '
          f'(filled with training mean popularity = {global_mean_pop:.1f}).')
else:
    print('No artist column found - skipping artist features.')

train_df.head(3)

## 4. A quick look at what correlates with popularity

Nothing here is used by the model directly - this is just to sanity-check
that the engineered features actually relate to the target before spending
time tuning a model on them.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

look_at = ['energy', 'danceability', 'loudness', 'acousticness',
           'dance_energy', 'energy_valence']

for i, feat in enumerate(look_at):
    if feat in train_df.columns:
        axes[i].scatter(train_df[feat], train_df['popularity'],
                        alpha=0.3, s=8, color='steelblue')
        valid = train_df[[feat, 'popularity']].dropna()
        z = np.polyfit(valid[feat], valid['popularity'], 1)
        xs = np.linspace(valid[feat].min(), valid[feat].max(), 100)
        axes[i].plot(xs, np.poly1d(z)(xs), color='tomato', linewidth=2)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('popularity')
        axes[i].set_title(f'popularity vs {feat}')

plt.suptitle('Feature vs popularity (train set only)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('models/eda_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Assembling the feature matrix

Keeping this list identical to what `app.py` builds for a manually-entered
or uploaded track (see `build_features()` in the app) - same names, same
order - is what lets the saved model plug straight into the app without any
translation step.

In [ ]:
FEATURES = [
    'energy', 'valence', 'danceability', 'loudness', 'acousticness',
    'tempo', 'speechiness', 'liveness',
    'energy_valence', 'dance_energy', 'acoustic_energy_gap',
    'loudness_norm', 'speech_ratio',
]
OPTIONAL = ['artist_mean_pop', 'artist_song_count']
FEATURES += [f for f in OPTIONAL if f in train_df.columns]

X_train = train_df[FEATURES].fillna(0)
y_train = train_df['popularity']
X_test  = test_df[FEATURES].fillna(0)
y_test  = test_df['popularity']

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features ({len(FEATURES)}): {FEATURES}')

## 6. Comparing a few models

A dummy mean-predictor as a sanity-check floor, then Random Forest, XGBoost,
and LightGBM under identical 5-fold cross-validation so the comparison is
fair. Whichever wins here is the one I tune further in the next step.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

MODELS = {
    'Dummy (mean baseline)': DummyRegressor(strategy='mean'),
    'Random Forest':         RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    'XGBoost':               XGBRegressor(n_estimators=200, random_state=RANDOM_STATE, verbosity=0),
    'LightGBM':              LGBMRegressor(n_estimators=200, random_state=RANDOM_STATE, verbose=-1),
}

results = []
for name, model in MODELS.items():
    cv_r2  = cross_val_score(model, X_train_s, y_train, cv=kf, scoring='r2')
    cv_mae = -cross_val_score(model, X_train_s, y_train, cv=kf, scoring='neg_mean_absolute_error')
    results.append({'Model': name, 'CV R2': cv_r2.mean(), 'CV R2 std': cv_r2.std(), 'CV MAE': cv_mae.mean()})
    print(f'{name:30s}  R2={cv_r2.mean():.3f} +/- {cv_r2.std():.3f}  MAE={cv_mae.mean():.2f}')

results_df = pd.DataFrame(results).sort_values('CV R2', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
ax.barh(results_df['Model'], results_df['CV R2'], xerr=results_df['CV R2 std'],
        color=colors, capsize=4, height=0.5)
ax.set_xlabel('Cross-validated R2')
ax.set_title('Model comparison (5-fold CV, train set only)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

BEST_MODEL_NAME = results_df.iloc[0]['Model']
print(f'\nBest model on CV: {BEST_MODEL_NAME}')

## 7. Tuning the winner with Optuna

Rather than hardcoding which model to tune, this picks whichever model won
the comparison above. In practice XGBoost or LightGBM usually take it on
tabular data like this, so both are supported here; Random Forest and the
dummy baseline aren't worth tuning further if they come out ahead, since
that would be unusual and worth investigating rather than optimizing.

Optuna searches 50 candidate hyperparameter combinations, scoring each with
3-fold CV, and keeps the best.

In [ ]:
def make_model(name, params=None):
    params = params or {}
    if name == 'XGBoost':
        return XGBRegressor(**params, random_state=RANDOM_STATE, verbosity=0)
    elif name == 'LightGBM':
        return LGBMRegressor(**params, random_state=RANDOM_STATE, verbose=-1)
    elif name == 'Random Forest':
        return RandomForestRegressor(**params, random_state=RANDOM_STATE)
    else:
        return DummyRegressor(strategy='mean')

def objective(trial):
    if BEST_MODEL_NAME == 'XGBoost':
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 10),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        }
    elif BEST_MODEL_NAME == 'LightGBM':
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 12),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        }
    else:
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth':    trial.suggest_int('max_depth', 3, 20),
        }
    model = make_model(BEST_MODEL_NAME, params)
    return cross_val_score(model, X_train_s, y_train, cv=3, scoring='r2').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest CV R2 from Optuna: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

## 8. Final model and test-set evaluation

This is the first and only time the test set is used to score the model -
everything up to this point (model choice, hyperparameters) was decided
using training-set cross-validation alone, so this number should be a fair
estimate of how the model performs on genuinely new tracks.

In [ ]:
best_model = make_model(BEST_MODEL_NAME, study.best_params)
best_model.fit(X_train_s, y_train)

y_pred = best_model.predict(X_test_s)

test_r2   = r2_score(y_test, y_pred)
test_mae  = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Test R2:   {test_r2:.4f}')
print(f'Test MAE:  {test_mae:.2f}')
print(f'Test RMSE: {test_rmse:.2f}')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.4, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual popularity')
ax.set_ylabel('Predicted popularity')
ax.set_title(f'Actual vs predicted  |  R2={test_r2:.3f}  MAE={test_mae:.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('models/actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Where does the model struggle?

An R2 of ~0.8 sounds good in isolation, but it's more useful to know *where*
the errors are concentrated. Plotting the residuals against the predicted
value shows whether the model is systematically off for certain kinds of
tracks (e.g. consistently underpredicting very popular songs) rather than
just scattered randomly.

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred, residuals, alpha=0.4, s=10, color='#c46a3a')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Predicted popularity')
axes[0].set_ylabel('Residual (actual - predicted)')
axes[0].set_title('Residuals vs predicted value')

axes[1].hist(residuals, bins=40, color='#c46a3a', alpha=0.8)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual distribution')

plt.tight_layout()
plt.savefig('models/residuals.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Mean residual: {residuals.mean():.2f} (close to 0 means no systematic bias)')

## 10. Explaining predictions with SHAP

Cross-validated R2 tells me the model is accurate; SHAP tells me *why* it
makes the predictions it does. This matters both for trusting the model and
for the "how to improve your track" advice in the app - that advice is only
credible if it's grounded in what actually moves the prediction.

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_s)
X_test_df = pd.DataFrame(X_test_s, columns=FEATURES)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('Global feature importance (mean |SHAP value|)')
plt.tight_layout()
plt.savefig('models/shap_global.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title("SHAP beeswarm - direction and magnitude of each feature's effect")
plt.tight_layout()
plt.savefig('models/shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

**Permutation importance**, as a second opinion on feature importance.
SHAP and permutation importance measure different things (SHAP explains
individual predictions and aggregates them; permutation importance measures
how much test-set accuracy drops when a feature is shuffled), so when they
roughly agree it's a good sign the "important" features are genuinely doing
the work, not just an artifact of how SHAP attributes credit in tree
models.

In [ ]:
perm = permutation_importance(
    best_model, X_test_s, y_test, n_repeats=15,
    random_state=RANDOM_STATE, scoring='r2'
)
perm_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(perm_df['feature'][::-1], perm_df['importance'][::-1],
        xerr=perm_df['std'][::-1], color='#7a9e9f', capsize=3)
ax.set_xlabel('Drop in test R2 when shuffled')
ax.set_title('Permutation importance')
plt.tight_layout()
plt.savefig('models/permutation_importance.png', dpi=120, bbox_inches='tight')
plt.show()

perm_df

One prediction, explained in full - picking a single test-set track and
showing exactly which features pushed its score up or down from the
average. This is the same kind of breakdown the app shows per-song.

In [ ]:
idx = 0  # change this to inspect a different test-set row
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_test_df.iloc[idx],
    feature_names=FEATURES,
), show=False)
plt.title(f'Predicted {y_pred[idx]:.0f} (actual: {y_test.iloc[idx]:.0f})')
plt.tight_layout()
plt.savefig('models/shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. How much should a single point estimate be trusted?

A predicted popularity of "72" implies more precision than the model
actually has. Looking at the spread of the test-set errors gives a rough
sense of the typical margin around a prediction, which I'm saving alongside
the model so the app can show a range instead of a bare number.

In [ ]:
error_margin = float(np.percentile(np.abs(residuals), 80))
print(f'80% of test predictions fall within +/- {error_margin:.1f} points of the true value.')

## 12. Logging the run to MLflow

This keeps a record of the parameters, metrics, and plots for this run so
future experiments can be compared against it. The model-logging step is
wrapped in a try/except - some environments (notably a fresh Colab session)
hit a dependency issue with MLflow's model serialization format that has
nothing to do with the model itself, and the actual file the app needs is
saved separately with `joblib` in the next step regardless.

In [ ]:
mlflow.set_experiment('music-popularity')

with mlflow.start_run(run_name=f'{BEST_MODEL_NAME.lower().replace(" ", "-")}-optuna'):
    mlflow.log_params(study.best_params)
    mlflow.log_param('model_type', BEST_MODEL_NAME)
    mlflow.log_param('features', FEATURES)
    mlflow.log_param('n_features', len(FEATURES))

    mlflow.log_metric('test_r2', test_r2)
    mlflow.log_metric('test_mae', test_mae)
    mlflow.log_metric('test_rmse', test_rmse)
    mlflow.log_metric('error_margin_80pct', error_margin)

    for artifact in ['actual_vs_predicted.png', 'shap_global.png',
                      'shap_beeswarm.png', 'residuals.png',
                      'permutation_importance.png']:
        path = f'models/{artifact}'
        if os.path.exists(path):
            mlflow.log_artifact(path)

    try:
        mlflow.sklearn.log_model(best_model, name='model')
    except Exception as e:
        print(f'MLflow model logging skipped (does not affect the saved .pkl below): {e}')

print('Run logged to MLflow. Run `mlflow ui` locally to browse it.')

## 13. Saving everything the Streamlit app needs

Four files, all under `models/`:
- `best_model.pkl` - the trained model
- `scaler.pkl` - the fitted `StandardScaler`, so the app scales new input the
  same way the training data was scaled
- `features.json` - the exact feature list and order the model expects
- `summary.json` - headline metrics the app displays in its stats row

These are all `app.py` reads at startup, so as long as they exist under
`models/`, the app works without any code changes.

In [ ]:
joblib.dump(best_model, 'models/best_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

with open('models/features.json', 'w') as f:
    json.dump(FEATURES, f)

summary = {
    'model':                 f'{BEST_MODEL_NAME} (Optuna-tuned)',
    'test_r2':               round(test_r2, 4),
    'test_mae':              round(test_mae, 2),
    'test_rmse':             round(test_rmse, 2),
    'error_margin_80pct':    round(error_margin, 2),
    'n_features':            len(FEATURES),
    'train_size':            len(X_train),
    'test_size':             len(X_test),
    # Used by the app as the fallback artist popularity for tracks from
    # artists not seen during training (uploads, manual entry).
    'global_mean_popularity': round(float(y_train.mean()), 2),
}
with open('models/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
for fname in sorted(os.listdir('models')):
    print(f'  models/{fname}')